In [19]:
import os
import re
import glob
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Ollama
import sys


In [20]:

def load_file(file_path):
    """外部ファイルを安全に読み込む関数"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"ファイルが見つかりません: {file_path}")
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

def extract_project_summary(text):
    """Slack投稿から案件概要部分を賢く切り出す改良版関数"""
    # パターン1: 記号（=== や ---）に囲まれている部分を優先して探す
    symbol_pattern = r'(?:={3,}|-{3,})'
    matches = [m.start() for m in re.finditer(symbol_pattern, text)]
    
    if len(matches) >= 2:
        # 記号が連続して重なっているケースを考慮し、最初と最後の記号の位置で切り出す
        start_idx = text.find('\n', matches[0]) + 1
        end_idx = matches[-1]
        summary = text[start_idx:end_idx].strip()
        if len(summary) > 50: # 最低限の文字数があれば採用
            return summary

    # パターン2: 記号がない場合、「案件」「概要」「必須」などのキーワードが出現する位置から末尾までを切り出す
    keyword_pattern = r'(【案件名】|案件：|概要：|【案件概要】|工程：)'
    match = re.search(keyword_pattern, text)
    if match:
        return text[match.start():].strip()
        
    # パターン3: 万が一どれにも引っかからない場合は、全文をそのまま渡す
    return text.strip()

# システムプロンプトを先読み
prompt_path = os.path.join("prompt", "system_prompt.txt")
prompt_template = load_file(prompt_path)
print("システムプロンプトの読み込み完了。")


システムプロンプトの読み込み完了。


In [21]:
# test_dataフォルダ内の全ての.txtファイルを取得
test_files = sorted(glob.glob(os.path.join("test_data", "*.txt")))

# 切り出し結果を辞書に格納
extracted_summaries = {}

for file_path in test_files:
    file_name = os.path.basename(file_path)
    raw_post_text = load_file(file_path)
    summary = extract_project_summary(raw_post_text)
    
    if summary:
        print(f"✅ {file_name}: 切り出し成功（{len(summary)} 文字）")
        extracted_summaries[file_name] = summary
    else:
        print(f"❌ {file_name}: 切り出し失敗")

# 例として、2つ目のデータの切り出し中身をプレビュー表示してみる
if "slack_post_2.txt" in extracted_summaries:
    print("\n--- slack_post_2.txt の切り出し結果プレビュー ---")
    print(extracted_summaries["slack_post_2.txt"][:200] + "...")


✅ slack_post_1.txt: 切り出し成功（551 文字）
✅ slack_post_2.txt: 切り出し成功（197 文字）
✅ slack_post_3.txt: 切り出し成功（482 文字）
✅ slack_post_4.txt: 切り出し成功（462 文字）

--- slack_post_2.txt の切り出し結果プレビュー ---
案件名：需要予測サービスの運用業務
工程：保守・運用

場所：浜松町（※リモート応相談）
期間：6月～

スキル：
必須）
　　・SQLを用いたデータ抽出、加工経験（2年以上）
　　・Pythonでの開発経験（2年以上）

人数：1名
外国籍：不可　
精算：140-180h　
面談：2回（web）　　　　　
年齢：45歳まで

備考：個人事業主、フリーランス不可　
　　・9:00-18:00...


In [22]:
# Ollamaのgemma2:9bモデルを初期化
llm = Ollama(model="gemma2:9b", temperature=0.0)
prompt = PromptTemplate(template=prompt_template, input_variables=["text"])
chain = prompt | llm

# 前のセルで切り出したデータを元にAI解析を実行
for file_name, summary in extracted_summaries.items():
    print(f"\n==========================================")
    print(f"🤖 AI解析実行中: {file_name}")
    print(f"==========================================")
    
    ai_result_json = chain.invoke({"text": summary})
    
    # 文字列の端にある余計な空白を削り、省略せずに強制出力する
    print(str(ai_result_json).strip())




🤖 AI解析実行中: slack_post_1.txt
{
    "project_name": "リサーチデータ利活用基盤の開発・保守（Python/SQL）",
    "summary": "リサーチ会社のデータ利活用基盤およびシステム基盤の開発・保守・運用業務です。PC・スマホのログやTV視聴ログなどのビッグデータを扱い、クレンジングや集計、それらを提供するWebアプリケーションの開発まで幅広く携わっていただきます。",
    "required_skills": "Pythonによるスクリプト作成、データ解析実務\nSQL（業務要件に基づいたクエリ作成、既存クエリの読解）\nLinux環境での詳細設計～テスト経験",
    "preferred_skills": "大量データ／ビッグデータのハンドリング経験（数億レコード規模の処理・集計の経験）\nSnowflake（他DWH製品（BigQuery、Redshift など）経験）\nTableau（ダッシュボード作成・データ可視化の実務経験）\nRuby、C#\nConfluence、JIRAなどAtlassian製品を用いたドキュメント管理、チケット管理\nバックエンド開発経験（バッチ処理、データ処理系が得意領域）",
    "period": "６月～長期",
    "location": "ひばりが丘",
    "hours": "09:00〜17:30（休憩1h）",
    "interview_count": "Web1回",
    "remarks": "週3〜4日リモート可\n140h-190h",
    "tools_and_languages": ["Python", "SQL", "Linux", "Snowflake", "BigQuery", "Redshift", "Tableau", "Ruby", "C#", "Confluence", "JIRA"]
}

🤖 AI解析実行中: slack_post_2.txt
{
    "project_name": "需要予測サービスの運用業務",
    "summary": "保守・運用",
    "required_skills": "SQLを用いたデータ抽出、加工経験（2年以上）\nPythonでの開発経験